# Day 01 · 遇見 Google ADK 2.0

> 第一部・新兵入伍　|　🧠 概念為主，附自製可執行實驗

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 01 - 遇見 Google ADK 2.0.md`

## 今天要學會

1. 說得出 ADK 的核心原語有哪些、彼此怎麼接
2. 解釋為什麼「讓 LLM 當總指揮」會失控——並**實際跑出那個失控**
3. 實測 ADK 2.0 的圖形化執行引擎（連單一 agent 都是圖上的節點）
4. 知道 1.x → 2.0 的破壞性變更

> 原文是純概念、零程式碼。本日的價值在於把那些論述**變成跑得出來的實驗**。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. ADK 的核心原語

先建立詞彙表。後面 29 天都在這張表上打轉：

| 原語 | 一句話 | 哪一天深入 |
|---|---|---|
| **Agent** | 有身分、有指令、有工具的執行單位 | Day 03 |
| **Tool** | Agent 的手腳，一個 Python 函式就是一個工具 | Day 06 |
| **Runner** | 執行引擎，把 agent 跑起來並吐出事件串流 | Day 04 |
| **Session** | 一段對話：`events`（發生過什麼）+ `state`（現在怎樣） | Day 09 |
| **Event** | 串流裡的一筆。工具呼叫、交棒、state 變更都是事件 | Day 12 |
| **App** | 應用層容器。壓縮、快取、plugins 都掛在這 | Day 10、11、12 |
| **Workflow** | ADK 2.0 的圖形化執行引擎 | Day 13、14 |

這一天不深入任何一個，只要知道它們存在、以及**彼此怎麼接**。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService


def add(a: int, b: int) -> dict:
    """把兩個整數相加。

    Args:
        a: 第一個數字。
        b: 第二個數字。
    """
    return {"result": a + b}


# Agent = 身分 + 指令 + 工具 + 模型
agent = LlmAgent(name="demo", model=get_model(), instruction="算術一律用工具。", tools=[add])

# App = 應用層設定
app = App(name="day01", root_agent=agent)

# Runner = 執行引擎，串起 App 與各種服務
runner = Runner(app=app, session_service=InMemorySessionService())

sid = await new_session(runner)
print(await ask(runner, "37 加 58 是多少？", session_id=sid, trace=True))

  🔧 [demo] 呼叫 add({'a': 37, 'b': 58})
  ↩️  [demo] add 回傳 {'result': 95}


  💬 [demo] 37 加 58 是 95。
37 加 58 是 95。


五個原語，一個 cell 就全部出現了。剩下 29 天都是在把每一個拆開來看。

## 2. 為什麼不能讓 LLM 當總指揮

原文的核心論點是：**抽象層級選錯了**。

很多人的第一版 agent 長這樣——把整個流程寫進 prompt，讓模型自己決定
「接下來要做什麼」。問題是模型沒有「我已經做過這件事」的可靠記憶，
它只是在猜下一個 token。

這不是理論。下面這個 agent 會真的卡住：

In [3]:
from shared import CallCounter

ATTEMPTS = []


def check_status(job_id: str) -> dict:
    """查詢任務的處理狀態。

    Args:
        job_id: 任務編號。
    """
    ATTEMPTS.append(job_id)
    # 模擬一個一直沒完成的後端任務
    return {"job_id": job_id, "status": "processing", "progress": f"{len(ATTEMPTS) * 3}%"}


stuck = LlmAgent(
    name="poller",
    model=get_model(),
    instruction=(
        "你要確認任務 JOB-1 是否完成。\n"
        "呼叫 check_status 查詢。如果 status 不是 'done'，就再查一次，"
        "直到完成為止。完成後回報結果。"
    ),
    tools=[check_status],
)

counter = CallCounter()
# ⚠️ 用了 app= 之後，plugins 要掛在 App 上，不能再傳給 Runner。
# 兩邊都給會得到：
#   ValueError: When app is provided, plugins should not be provided
#               and should be provided in the app instead.
runner_stuck = Runner(
    app=App(name="day01", root_agent=stuck, plugins=[counter]),
    session_service=InMemorySessionService(),
)
sid = await new_session(runner_stuck)

# ⚠️ 先裝煞車再點火。沒有 max_llm_calls 的話，這個 agent 會一路跑到 ADK 的
# 預設上限為止——那個數字大到足以讓你的免費額度在一個 cell 裡蒸發。
from google.adk.agents.run_config import RunConfig
from google.genai import types

message = types.Content(role="user", parts=[types.Part(text="確認 JOB-1 完成了沒")])
try:
    async for _event in runner_stuck.run_async(
        user_id="student", session_id=sid, new_message=message,
        run_config=RunConfig(max_llm_calls=8),
    ):
        pass
    print("它自己停下來了")
except Exception as exc:
    print(f"❌ 被上限擋下來：{type(exc).__name__}")
    print(f"   {str(exc)[:120]}")

print(f"\n工具被呼叫了 {len(ATTEMPTS)} 次")
print("📊", counter.report())

❌ 被上限擋下來：LlmCallsLimitExceededError
   Max number of llm calls limit of `8` exceeded

工具被呼叫了 8 次
📊 模型呼叫 9 次；工具呼叫 8 次：check_status, check_status, check_status, check_status, check_status, check_status, check_status, check_status


不管它是跑滿上限被擋下來、還是自己放棄，重點是一樣的：

> **「什麼時候該停」這件事，你交給了一個在猜下一個字的模型。**

而它每猜一次就是一次 API 呼叫、一次帳單。

### `max_llm_calls`：唯一的煞車

剛剛那個 `max_llm_calls=8` 就是安全閥。它是 **`RunConfig` 的欄位**，
要在 `run_async()` 時傳，不是設在 agent 上。

把它調得更緊，看它多快撞牆：

In [4]:
ATTEMPTS.clear()
runner_capped = Runner(
    app=App(name="day01", root_agent=stuck),
    session_service=InMemorySessionService(),
)
sid = await new_session(runner_capped)

try:
    async for _ in runner_capped.run_async(
        user_id="student",
        session_id=sid,
        new_message=message,
        run_config=RunConfig(max_llm_calls=3),  # ← 煞車
    ):
        pass
except Exception as exc:
    print(f"被擋下來了：{type(exc).__name__}")
    print(str(exc)[:200])

print(f"\n工具呼叫次數: {len(ATTEMPTS)}（被 max_llm_calls=3 限制住）")

被擋下來了：LlmCallsLimitExceededError
Max number of llm calls limit of `3` exceeded

工具呼叫次數: 3（被 max_llm_calls=3 限制住）


> ⚠️ **`max_llm_calls` 不保護 BIDI 串流**。這個護欄只涵蓋 SSE 與
> `run_async`。Live API 的雙向串流不受它保護——細節在 Day 22。

### 正確的做法：把「流程」交給程式，「判斷」交給模型

同樣的需求，用 ADK 2.0 的 `Workflow` 寫。輪詢幾次、什麼時候放棄，
都是**確定性的 Python 程式碼**；模型只負責它擅長的事。

In [5]:
from google.adk import Workflow
from google.adk.workflow import START, node
from google.adk.agents.context import Context

POLLS = []


@node
def poll_until_done(ctx: Context) -> dict:
    """最多輪詢 5 次，這是程式的決定，不是模型的。"""
    for i in range(5):
        POLLS.append(i)
        status = {"job_id": "JOB-1", "status": "done" if i == 3 else "processing"}
        if status["status"] == "done":
            # 寫進 state，下游的 reporter 才能用 {job_result?} 讀到。
            # 節點之間的資料是走 state，不是走回傳值——Day 13/14 會細講。
            ctx.state["job_result"] = status
            ctx.route = "done"
            return status
    ctx.state["job_result"] = {"status": "timeout", "polls": len(POLLS)}
    ctx.route = "timeout"
    return ctx.state["job_result"]


reporter = LlmAgent(
    name="reporter",
    model=get_model(),
    instruction="根據 state 裡的 {job_result?}，用一句繁體中文向使用者說明任務結果。",
)


@node
def escalate() -> str:
    return "輪詢逾時，已轉人工處理。"


poll_flow = Workflow(
    name="poll_flow",
    edges=[
        (START, poll_until_done),
        (poll_until_done, {"done": reporter, "timeout": escalate}),
    ],
)

In [6]:
from google.adk.runners import InMemoryRunner

runner_wf = InMemoryRunner(agent=poll_flow, app_name="day01")
sid = await new_session(runner_wf)
msg = types.Content(role="user", parts=[types.Part(text="確認 JOB-1")])

async for event in runner_wf.run_async(user_id="student", session_id=sid, new_message=msg):
    out = getattr(event, "output", None)
    if out is not None:
        print(f"  ▪ [{event.author}] {out}")
    elif event.is_final_response() and event.content:
        text = "".join(p.text or "" for p in event.content.parts if p.text)
        if text:
            print(f"  💬 {text.strip()}")

print(f"\n輪詢次數: {len(POLLS)}（由程式決定，完全可預期）")

  ▪ [poll_flow] {'job_id': 'JOB-1', 'status': 'done'}


  💬 任務 JOB-1 已順利完成。

輪詢次數: 4（由程式決定，完全可預期）


對照一下：

| | LLM 當總指揮 | 流程交給程式 |
|---|---|---|
| 輪詢次數 | 不可預期 | 寫死 5 次 |
| 什麼時候放棄 | 模型的心情 | `ctx.route = "timeout"` |
| 成本 | 每次判斷都是一次 API 呼叫 | 輪詢完全不花 token |
| 重跑結果 | 每次都可能不同 | 一樣 |

**這就是 ADK 存在的理由**：不是讓你更容易呼叫 LLM，
而是讓你把「流程」跟「判斷」分開。

## 3. 📌 文章沒講到的補充：ADK 2.0 真的是圖形引擎嗎

原文說 2.0「從階層式執行器換成圖形化執行引擎」。這句話聽起來像行銷詞。
我們直接驗證。

**實驗**：跑一個**沒有任何 workflow、沒有 sub_agent** 的孤零零 `LlmAgent`，
讓它失敗，然後看 traceback 經過哪些檔案。

In [7]:
import traceback

lonely = LlmAgent(name="lonely", model="gemini-this-model-does-not-exist", instruction="hi")

try:
    await run_once(lonely, "hello")
except Exception as exc:
    frames = traceback.extract_tb(exc.__traceback__)
    print(f"例外型別: {type(exc).__module__}.{type(exc).__name__}\n")
    print("traceback 裡跟 workflow 引擎相關的層：")
    for f in frames:
        short = f.filename.split("site-packages/")[-1]
        if "workflow" in short:
            print(f"  {short}:{f.lineno}  in {f.name}")

例外型別: google.genai.errors.ClientError

traceback 裡跟 workflow 引擎相關的層：
  google/adk/workflow/_node_runner.py:136  in run
  google/adk/workflow/_node_runner.py:274  in _execute_node
  google/adk/workflow/_node_runner.py:288  in _run_node_loop
  google/adk/workflow/_base_node.py:170  in run
  google/adk/workflow/_llm_agent_wrapper.py:484  in run_llm_agent_as_node


結果很明確。一個**單獨的、沒有任何 workflow 的** `LlmAgent`，
它的呼叫堆疊裡有這些東西：

| 出現的檔案 / 函式 | 說明 |
|---|---|
| `workflow/_node_runner.py` | 節點執行器——連單一 agent 都由它驅動 |
| `workflow/_base_node.py` | 基礎節點 |
| `workflow/_llm_agent_wrapper.py` 的 **`run_llm_agent_as_node`** | 函式名字就寫得很白：**把 LlmAgent 當成節點來跑** |

**每一個 agent 都是圖上的一個節點**，只是單一 agent 的圖只有一個節點。
這不是行銷詞，是實際的架構。

（ADK 內部另外會用 `DynamicNodeFailError` 包裝節點失敗並寫進 log，
但最後往外拋的仍然是原始例外——這裡是 `ClientError`——所以你的
`except` 寫法不用改。）

這件事的實際後果：你在 Day 13 學 `Workflow` 時，學的不是「另一種寫法」，
而是**把一直都在的那張圖顯式地畫出來**。

## 4. 📌 補充：1.x → 2.x 的實際差異

光說「有破壞性變更」沒有用。下面直接用 import 驗證哪些 API 是 2.x 才有的：

In [8]:
CHECKS = {
    "Workflow 圖形引擎": "from google.adk import Workflow",
    "圖節點工具": "from google.adk.workflow import START, JoinNode, node",
    "人機協作節點": "from google.adk.events import RequestInput",
    "App 容器": "from google.adk.apps import App",
    "可恢復執行": "from google.adk.apps import ResumabilityConfig",
    "上下文壓縮": "from google.adk.apps.app import EventsCompactionConfig",
    "內容快取": "from google.adk.agents.context_cache_config import ContextCacheConfig",
    "Agent Skills": "from google.adk.tools.skill_toolset import SkillToolset",
    "Managed Agents": "from google.adk.agents import ManagedAgent",
}

for label, stmt in CHECKS.items():
    try:
        exec(stmt)
        print(f"  ✅ {label}")
    except Exception as exc:
        print(f"  ❌ {label}: {type(exc).__name__}")

  ✅ Workflow 圖形引擎
  ✅ 圖節點工具
  ✅ 人機協作節點
  ✅ App 容器
  ✅ 可恢復執行
  ✅ 上下文壓縮
  ✅ 內容快取
  ✅ Agent Skills
  ✅ Managed Agents


這些全部是 **ADK 2.x 才有的**。如果你的環境是 1.x，上面會有一片紅色，
而且 Day 13、14、15、24 整天都跑不起來。

### 版本快速自檢

In [9]:
version = google.adk.__version__
major = int(version.split(".")[0])
print(f"目前版本: {version}")
if major >= 2:
    print("✅ 可以跑完 30 天全部內容")
else:
    print("❌ 請執行 `uv sync` 升級到 2.x，否則 Day 13/14/15/24 無法執行")

目前版本: 2.8.0
✅ 可以跑完 30 天全部內容


## 5. 常見錯誤與踩坑

**坑 1：以為換成 ADK 就不會失控**

不會。ADK 給的是「可以不失控的工具」，但你得選擇用它。
第 2 節那個 `stuck` agent 完全是合法的 ADK 程式碼——它一樣會失控。

**坑 2：`max_llm_calls` 以為是全域保險**

它不涵蓋 BIDI 串流（Day 22）。而且它是 `RunConfig` 的欄位，
要在 `run_async()` 時傳，不是設在 agent 上。

**坑 3：`app=` 和 `plugins=` 不能同時給 Runner**

用了 `Runner(app=App(...))` 之後，plugin 要掛在 `App` 上。兩邊都給會拋
`ValueError: When app is provided, plugins should not be provided`。
只有 `Runner(agent=...)` 那條舊路才在 Runner 上收 plugins。

**坑 4：混用兩套資料流機制**

`Workflow` 的節點輸出在 `event.output`；
`SequentialAgent` 那一套用 `output_key` 寫進 state、`{key?}` 讀出來。
兩者不同，混用會拿到 `None`。Day 14 會完整比較。

In [10]:
# 坑 3 的實例：對 Workflow 的節點輸出用 is_final_response() 是抓不到的
runner_demo = InMemoryRunner(agent=poll_flow, app_name="day01")
sid = await new_session(runner_demo)
msg = types.Content(role="user", parts=[types.Part(text="確認 JOB-1")])

node_outputs, final_texts = [], []
async for event in runner_demo.run_async(user_id="student", session_id=sid, new_message=msg):
    if getattr(event, "output", None) is not None:
        node_outputs.append(event.output)
    if event.is_final_response() and event.content:
        t = "".join(p.text or "" for p in event.content.parts if p.text)
        if t:
            final_texts.append(t)

print(f"event.output 抓到 {len(node_outputs)} 筆：")
for o in node_outputs:
    print(f"   ▪ {o}")
print(f"\nis_final_response() 抓到 {len(final_texts)} 筆：")
for t in final_texts:
    print(f"   ▪ {t.strip()}")

print("\n→ 兩者抓到的是**完全不同的東西**：")
print("  event.output          = @node 函式的回傳值（輪詢結果）")
print("  is_final_response()   = LLM agent 的文字回應（給人看的那句話）")
print("  只用其中一個，就會漏掉另一半。")

event.output 抓到 1 筆：
   ▪ {'job_id': 'JOB-1', 'status': 'done'}

is_final_response() 抓到 1 筆：
   ▪ 任務 JOB-1 已順利完成。

→ 兩者抓到的是**完全不同的東西**：
  event.output          = @node 函式的回傳值（輪詢結果）
  is_final_response()   = LLM agent 的文字回應（給人看的那句話）
  只用其中一個，就會漏掉另一半。


## 6. 動手練習

1. 把第 2 節 `check_status` 改成「第 3 次呼叫就回 done」，
   看 `stuck` agent 這次會不會正常結束。它每次都一樣嗎？跑五次看看。
2. 把 `poll_until_done` 的迴圈上限改成 3，讓它走 `timeout` 分支，
   確認 `escalate` 節點有被觸發。
3. 用第 3 節的 traceback 技巧，觀察一個 `SequentialAgent` 失敗時
   會經過哪些 workflow 檔案——它跟單一 agent 一樣嗎？

## 本日回顧

- **ADK 的核心原語**：Agent / Tool / Runner / Session / Event / App / Workflow。
- **不要讓模型決定流程**。「什麼時候停」交給在猜下一個字的模型，
  就是把成本和正確性一起交出去。
- **`max_llm_calls` 是煞車**，但它不保護 BIDI 串流。
- **ADK 2.0 的圖形引擎是真的**：單一 `LlmAgent` 的 traceback 也會穿過
  `workflow/_node_runner.py`，失敗時拋 `DynamicNodeFailError`。
- **1.x 跑不了這 30 天**：`Workflow`、`App`、`RequestInput` 等都是 2.x 才有。
- **圖的節點之間走 state 傳資料**，不是走回傳值；plugins 掛在 `App` 不是 `Runner`。

---
**下一天 → `../day02_environment_and_cli/`**